In [ ]:
!pip install aiohttp aiofiles beautifulsoup4 pandas requests

In [15]:
import asyncio
import io
import aiohttp
import csv
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
import requests

# --- ⚙️ CONFIGURATION ⚙️ ---
TOURNAMENTS = [
    {"id": "4626469", "name": "PBL", "date": "09/06/2025"},
    {"id": "9460497", "name": "UBL 1-1", "date": "11/08/2025"},
    {"id": "608123", "name": "UBL 1-2", "date": "11/15/2025"},
    {"id": "8762799", "name": "UBL 1-3", "date": "11/23/2025"},
    {"id": "5669429", "name": "UBL 1-4", "date": "11/29/2025"},
    {"id": "7826390", "name": "UBL 1-5", "date": "11/30/2025"}
]

# Set a concurrency limit. 10 is a safe start to avoid getting IP banned.
MAX_CONCURRENT_REQUESTS = 10 

async def fetch_ubl_player(semaphore, player_dict, tid, name, date, base_url, profile_url):
    """Async worker to fetch a single player's data using dynamic cookie injection."""
    current_id = player_dict['id']
    dropdown_txt = player_dict['dropdown_text']

    async with semaphore:
        await asyncio.sleep(random.uniform(0.2, 0.7))
        
        # 💉 CONSTRUCT THE EXACT COOKIE FROM YOUR SCREENSHOT
        # Name: TCG + Tournament ID
        # Value: ShikibetsuNo + Player ID + &Visitor=
        cookie_string = f"TCG{tid}=ShikibetsuNo={current_id}&Visitor="
        
        headers = {
            'User-Agent': 'Mozilla/5.0', 
            'Referer': base_url,
            'Cookie': cookie_string
        }
        
        # Create session with our spoofed headers
        async with aiohttp.ClientSession(headers=headers) as session:
            try:
                # Go straight to the profile page
                async with session.get(profile_url) as profile_resp:
                    html_bytes = await profile_resp.read()
                    
                soup = BeautifulSoup(html_bytes, 'html.parser', from_encoding='utf-8')

                found_name = None
                target_str = f"'{current_id}'"
                target_row = soup.find('tr', onclick=lambda x: x and target_str in x)

                if target_row:
                    cells = target_row.find_all('td')
                    if len(cells) >= 2:
                        found_name = cells[1].get_text(strip=True)

                if found_name:
                    return [dropdown_txt, found_name, current_id, tid, name, date]
                
                return None

            except Exception as e:
                print(f"⚠️ Error on ID {dropdown_txt}: {e}")
                return None

async def scrape_ubl_tournament(tid, name, date, csv_filename):
    print(f"📊 Scraping Tournament: {name} ({date}) with ID: {tid}")

    BASE_URL = f"https://tcg.sfc-jpn.jp/loginnum.asp?tid={tid}&MMP=&flu=&Exclusive=0"
    PROFILE_URL = f"https://tcg.sfc-jpn.jp/tour.asp?tid={tid}&kno=1&znt=1&MMP=&flu=&Exclusive=0"
    
    dropdown_results = []
    
    # Fetching the dropdown list can be done synchronously or with a quick one-off async session
    async with aiohttp.ClientSession() as session:
        try:
            async with session.get(BASE_URL) as resp:
                html_bytes = await resp.read()
                soup = BeautifulSoup(html_bytes, 'html.parser')

                select = soup.find('select', attrs={'name': 'SelectShikibetsuNo'})
                if not select:
                    print("❌ Error: Dropdown not found.")
                    return

                for opt in select.find_all('option'):
                    val = opt.get('value')
                    text = opt.get_text(strip=True)
                    if val and val.strip():
                        dropdown_results.append({'id': val, 'dropdown_text': text})
        except Exception as e:
            print(f"❌ Connection Error: {e}")
            return

    print(f"Found {len(dropdown_results)} players in dropdown. Starting async extraction...")

    # Set up the semaphore to limit concurrency
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    
    # Create a list of tasks for all players
    tasks = [
        fetch_ubl_player(semaphore, p, tid, name, date, BASE_URL, PROFILE_URL) 
        for p in dropdown_results
    ]

    # Run all tasks concurrently and wait for them to finish
    results = await asyncio.gather(*tasks)

    # Filter out any None values (failed requests or missing names)
    players_in_tournament = [res for res in results if res is not None]

    # Save to CSV
    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        if csvfile.tell() == 0:
            writer.writerow(['Dropdown Text', 'Player Name', 'Player ID', 'Tournament ID', 'Tournament Name', 'Tournament Date'])
        writer.writerows(players_in_tournament)
        print(f"💾 Saved {len(players_in_tournament)} records to {csv_filename}")

def clean_csv_line(line: str) -> str:
    """
    Detects if a row has extra commas due to "Last Name, First Name" formatting 
    and wraps the split names in double quotes so Pandas parses them as a single string.
    """
    parts = line.split(',')
    
    # Normal rows have exactly 7 columns (which means 6 commas)
    if len(parts) <= 7:
        return line
        
    # If there is 1 extra comma (8 columns)
    if len(parts) == 8:
        # Check if the extra comma is in "Your name" by looking at where the numeric TtlScore got pushed
        if parts[4].strip().isdigit(): 
            parts[2] = f'"{parts[2]},{parts[3]}"'
            del parts[3]
        # Check if the extra comma is in "Opponent"
        elif parts[6].strip().isdigit():
            parts[4] = f'"{parts[4]},{parts[5]}"'
            del parts[5]
            
    # If BOTH "Your name" and "Opponent" have extra commas (9 columns)
    elif len(parts) == 9:
        # Fix Opponent first (working backwards prevents index shifting)
        parts[5] = f'"{parts[5]},{parts[6]}"'
        del parts[6]
        # Then fix Your name
        parts[2] = f'"{parts[2]},{parts[3]}"'
        del parts[3]
        
    return ','.join(parts)

async def scrape_pbl_mbl_tournament(tid, name, date, csv_filename):
    CSV_URL = f"https://tcg.sfc-jpn.jp/tourcsvdl.asp?tid={tid}&kno=3&blk=&znt=0&Sort=Table&Order=Asc&CharCode=UTF-8&MMP=1&flu=&CurScr=0"

    print(f"\nFetching and cleaning CSV data from {CSV_URL}...")
    try:
        # 1. Fetch raw text manually
        response = requests.get(CSV_URL)
        response.encoding = 'utf-8'  # Ensure correct encoding
        response.raise_for_status()
        raw_text = response.text
        
        # 2. Clean the text line-by-line
        cleaned_lines = [clean_csv_line(line) for line in raw_text.splitlines()]
        cleaned_text = '\n'.join(cleaned_lines)
        
        # 3. Feed the cleaned string to Pandas using StringIO
        df = pd.read_csv(io.StringIO(cleaned_text))
        
    except Exception as e:
        print(f"Error fetching or parsing the CSV: {e}")
        return

    # Check if the required columns exist
    if not {'No.', 'Your name'}.issubset(df.columns):
        print("Error: The CSV does not contain the required 'No.' and 'Your name' columns.")
        return

    # Create a fast lookup dictionary: {'No.': 'Your name'}
    # e.g., {'ph38012733': 'Aldrin Almodiel', ...}
    # We skip deduplication since you confirmed there are no duplicates in this file
    csv_updates_lookup = dict(zip(df['No.'], df['Your name']))

    players_in_tournament = [
        [index, v, k, tid, name, date] 
        for index, (k, v) in enumerate(csv_updates_lookup.items(), start=1)
    ]

    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        if csvfile.tell() == 0:
            writer.writerow(['Dropdown Text', 'Player Name', 'Player ID', 'Tournament ID', 'Tournament Name', 'Tournament Date'])
        writer.writerows(players_in_tournament)
        print(f"💾 Saved {len(players_in_tournament)} records to {csv_filename}")

async def scrape_tournament(tid, name, date, csv_filename):
    if "UBL" in name:
        await scrape_ubl_tournament(tid, name, date, csv_filename)
    elif "PBL" in name or "MBL" in name:
        await scrape_pbl_mbl_tournament(tid, name, date, csv_filename)
    else:
        print(f"⚠️ No scraper defined for tournament type: {name}")

async def run_scraper():
    print("--- 🚀 Starting Optimized Async Scraper ---")
    
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    csv_filename = f"tournament_players_{timestamp}.csv"
    
    for tournament in TOURNAMENTS:
        print(f"\n🎯 Processing Tournament: {tournament['name']} ({tournament['date']})")
        await scrape_tournament(tournament['id'], tournament['name'], tournament['date'], csv_filename)

if __name__ == "__main__":
    await run_scraper()

--- 🚀 Starting Optimized Async Scraper ---

🎯 Processing Tournament: PBL (09/06/2025)

Fetching and cleaning CSV data from https://tcg.sfc-jpn.jp/tourcsvdl.asp?tid=4626469&kno=3&blk=&znt=0&Sort=Table&Order=Asc&CharCode=UTF-8&MMP=1&flu=&CurScr=0...
💾 Saved 799 records to tournament_players_20260405-195909.csv

🎯 Processing Tournament: UBL 1-1 (11/08/2025)
📊 Scraping Tournament: UBL 1-1 (11/08/2025) with ID: 9460497
Found 258 players in dropdown. Starting async extraction...


CancelledError: 